In [ ]:
%pip install -e ../../../
%pip install plotly
%pip install nbformat
%pip install kaleido

In [ ]:
import itertools
import numpy as np
from numpy.random import default_rng
RNG = default_rng(12804)
import pandas as pd

In [ ]:
from water_futures_battle.nrw_model.enums import NRWClass
from water_futures_battle.jurisdictions.enums import MunicipalitySize

# We calculate the base cost as the cost to replace 1% of the pipes in year 2000 
# with the smallest diameter possible. 1050 km at 45 €/m -> 47 M€ 
# so we do 47M€/year divide by 105000 km to get the base cost. Simplified it is just the cost 
# of the smallest pipe per km divide by 100...
# Truth is cost should be slightly higher than installing the smallest newset pipe.
# But we are usign the costs in 2025, so consdering 2025 of inflation is already good
base_cost = 450
# Cost increases if nrw class decreases, easier to get better when you are terrible.
# To tune it I take the average value of the class (or the highest age for class A 
# and the lowest for class E). That average gives you a replacement rate.
# We express them all as ratio of class C (which has 1%, i.e. the current real rate).
nrw_class_factors = {c: v for c, v in zip(
    [c.name for c in NRWClass],
    [2.0, 1.25, 1.0, 0.96, 0.83]
)}
# Cost increases if municipality is bigger, more problems and equipment
size_class_factors = {c: v for c, v in zip(
    [c.name for c in MunicipalitySize],
    [1.0, 1.04, 1.12, 1.2]
)}

NRW_INTERVENTION_COLUMNS = [f"NL0000-{c[0].name}-{c[1].name}" for c in itertools.product(NRWClass, MunicipalitySize)]

df = pd.DataFrame(columns=NRW_INTERVENTION_COLUMNS, index=pd.DatetimeIndex(["2000-01-01"], name='timestamp'))

for c, col_name in zip(itertools.product(NRWClass, MunicipalitySize), NRW_INTERVENTION_COLUMNS):
    nrw_class = c[0].name
    muni_size = c[1].name
    df.loc["2000-01-01", col_name] = base_cost * nrw_class_factors[nrw_class] * size_class_factors[muni_size]
    

In [ ]:
from water_futures_battle.core.base_model import DynamicProperties

nrw_dps = DynamicProperties('nrw_model-dynamic_properties')
nrw_dps['nrw_intervention-unit_cost'] = df.copy()
nrw_dps.dump('..')

## Figure for the paper

In [ ]:
from water_futures_battle.nrw_model.enums import NRWClass, _NRW_CLASSES_BOUNDS_LIST

values = {}
for c in NRWClass:
    values[c.name] = c.sample_demand(n_points=100, RNG=RNG)

import plotly.graph_objects as go
import pandas as pd

# Prepare data for box plot
class_names = list(values.keys())
all_samples = [values[c] for c in class_names]

fig = go.Figure()
for cname, samples in zip(class_names, all_samples):
    fig.add_trace(go.Box(
        y=samples,
        name=cname,
        boxmean=True,
        marker_color='blue',
        line_color='blue',
        fillcolor='rgba(0,0,255,0.2)',
        boxpoints=False
    ))

# Add horizontal lines at NRW class bounds (except first and last)
for bound in _NRW_CLASSES_BOUNDS_LIST[1:-1]:
    fig.add_shape(
        type='line',
        x0=-0.5, x1=len(class_names)-0.5,
        y0=bound, y1=bound,
        line=dict(color='grey', width=1, dash='dash'),
        xref='x', yref='y',
        layer='below'
    )

# Set y-axis ticks and labels at the bounds, with labels at the tick positions
bound_labels = [f"{b:.0f}" for b in _NRW_CLASSES_BOUNDS_LIST]
fig.update_layout(
    xaxis_title="NRW Class",
    yaxis_title="NRW Demand Factor [m^3/km/day] ",
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=40, r=40, t=60, b=40),
    xaxis=dict(
        showline=True,
        linewidth=2,
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        showline=True,
        linewidth=2,
        linecolor='black',
        mirror=True,
        tickmode='array',
        tickvals=_NRW_CLASSES_BOUNDS_LIST,
        ticktext=bound_labels
    )
)
fig.show()

In [ ]:
# Save figure to PNG with LaTeX y-axis label
import plotly.io as pio

# Update y-axis title to LaTeX
fig.update_layout(yaxis_title=r"$\mathrm{NRW~Demand~Factor}~[m^3/km/day]$")

# Save to PNG (requires kaleido)
pio.write_image(fig, "leak_demand.png", format="png", scale=3)